# ND1 · 1.3–1.5. Neuroniniai tinklai, CNN ir perkėlimo mokymasis

**Liutauras Turauskis · DIPf-24**

Viename užrašyne sujungtos visos penkios dėstytojo pavyzdžiais paremtos dalys:

| Užduotis | Turinys | Pasirinkti duomenys |
|---|---|---|
| 1.3.1 | TensorFlow neuroninis tinklas | UCI Bank Marketing |
| 1.3.2 | TensorFlow Decision Forests | Tas pats banko duomenų padalijimas |
| 1.4 | CNN, filtrai, rankinė koreliacija ir MLP | Flowers Photos |
| 1.5.1 | Gatavų modelių klasifikavimas, iškarpa, aptikimas ir segmentavimas | Gėlės nuotrauka |
| 1.5.2 | Naujos galvos mokymas ir fine-tuning | Penkios gėlių klasės |

**Paleidimas:** naujoje Google Colab sesijoje pasirinkti GPU, jei prieinamas, ir **Runtime → Run all**.
Ląsteles vykdyti nuo pradžios: banko duomenys paruošiami 1.3.1, o vaizdai – 1.4 dalyje.
Vėlesnės dalys naudoja tuos pačius duomenis ir jų padalijimą. API rakto ir savų failų nereikia.

Bendros bibliotekos įdiegiamos vieną kartą. TFDF vykdomas atskiroje suderinamoje Linux / Python 3.12 aplinkoje;
Colab branduolio perkrauti tarp skyrių nereikia. Pirmas duomenų ir parengtų modelių atsisiuntimas užtrunka.
Kiekvienos užduoties eksportai saugomi atskirame `rezultatai/` pakatalogyje.

**Patikros būsena:** parengiant patikrinta Python sintaksė, ląstelių struktūra ir jų priklausomybės.
Visas mokymas šioje aplinkoje nebuvo vykdytas; tikrų mokymo išvesčių ši kopija dar neturi.
Rodikliai ir grafikai sugeneruojami paleidus Colab, o ne įrašomi iš anksto.
Baigus išsaugoti šį užrašyną į GitHub su išvestimis.


## Bendras aplinkos paruošimas


In [ ]:
%pip -q install "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" requests pillow


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report)
from IPython.display import display
SEED = 42
OUT = Path('rezultatai')
OUT.mkdir(exist_ok=True)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
RESULTS_ROOT = Path('rezultatai')
OUT = RESULTS_ROOT / '1_3'
OUT.mkdir(parents=True, exist_ok=True)


## 1.3.1. Neuroninis tinklas banko klientų duomenims

Pritaikyta pagal dėstytojo pateiktą `AdventureWorksDW_NN.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Dviračio pirkimo prognozė pakeista indėlio užsakymo prognoze; išsaugotas skaitinių ir kategorinių požymių apdorojimas, tankieji sluoksniai, mokymas ir išsaugojimas.


### 1. Kiti verslo duomenys ir prognozės tikslas

Originale buvo „AdventureWorks“ klientai ir `BikeBuyer` (ar pirko dviratį).
Čia naudojami **UCI Bank Marketing** duomenys: ar klientas užsisakė terminuotąjį indėlį (`y`).
Tai viešas pasirinktas, o ne mano paties surinktas rinkinys. Originalūs autoriai:
S. Moro, P. Rita, P. Cortez; [UCI šaltinis ir CC BY 4.0 licencija](https://archive.ics.uci.edu/dataset/222/bank+marketing).
Verslo rinkinys taip pat platinamas [Kaggle](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset);
čia naudojamas UCI originalas, kad nereikėtų Kaggle prisijungimo.

Pasirenkama iki 8 000 įrašų iš `bank-full.csv`, išlaikant klasių proporcijas.
`duration` pašalinamas: skambučio trukmė nežinoma prieš skambutį ir lemtų informacijos nutekėjimą.
Likę kontaktų bei ankstesnės kampanijos požymiai naudojami kaip turima kliento informacija.
Tai mokomasis atsitiktinio padalijimo eksperimentas; būsimos kampanijos kokybei reikėtų laikinio padalijimo.


In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
r = requests.get(URL, timeout=90)
r.raise_for_status()
# UCI archyve bank-full.csv gali būti papildomame bank.zip archyve.
def read_bank_csv(blob):
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        names = archive.namelist()
        csv_name = next((n for n in names if n.endswith('bank-full.csv')), None)
        if csv_name:
            return pd.read_csv(archive.open(csv_name), sep=';')
        nested = next((n for n in names if n.rsplit('/', 1)[-1] == 'bank.zip'), None)
        if nested:
            return read_bank_csv(archive.read(nested))
    raise ValueError('UCI archyve nerastas bank-full.csv; patikrinkite šaltinį.')

raw = read_bank_csv(r.content)
assert set(raw['y']) == {'yes', 'no'}
df = raw.drop(columns=['duration']).copy()
df['target'] = df.pop('y').map({'no': 0, 'yes': 1}).astype('int32')
if len(df) > 8000:
    df, _ = train_test_split(df, train_size=8000, stratify=df['target'], random_state=SEED)
train, remaining = train_test_split(df, test_size=0.30, stratify=df['target'], random_state=SEED)
val, test = train_test_split(remaining, test_size=0.50, stratify=remaining['target'], random_state=SEED)
assert set(train.index).isdisjoint(val.index) and set(train.index).isdisjoint(test.index)
assert set(val.index).isdisjoint(test.index)
assert 'target' not in train.drop(columns='target') and 'duration' not in df
display(df.head())
display(pd.DataFrame({k: {'įrašai': len(v), 'indėlis_1_dalis': v.target.mean()}
                      for k, v in {'mokymas': train, 'validavimas': val, 'testas': test}.items()}).T)
display(df.dtypes.rename('tipas').to_frame())
print('Trūkstamos reikšmės:', int(df.isna().sum().sum()))
# unknown paliekamas kaip atskira kategorija; pdays=-1 reiškia, kad anksčiau nekontaktuota.
assert not df.isna().any().any()
for name, part in [('train', train), ('val', val), ('test', test)]:
    part.to_csv(OUT / f'bank_{name}.csv', index=False)


### 2. Duomenų paruošimas tinklui

Originalo `df_to_dataset` turėjo riziką palikti tikslą požymių žodyne. Čia `target` pirmiausia
atskiriamas ir į modelio įvestis nepatenka. Skaitinius stulpelius standartizuoja `Normalization`,
tekstines kategorijas `StringLookup` paverčia one-hot vektoriais. Abiejų sluoksnių `adapt`
mato **tik mokymo** duomenis; validavimo bei testo statistika nenaudojama.
Nežinoma būsimo įrašo kategorija turi atskirą OOV vietą.


In [ ]:
numeric = train.drop(columns='target').select_dtypes(include='number').columns.tolist()
categorical = train.drop(columns='target').select_dtypes(exclude='number').columns.tolist()

def feature_dict(frame):
    features = frame.drop(columns='target', errors='ignore')
    assert 'target' not in features
    return {c: tf.convert_to_tensor(features[c].to_numpy(dtype='float32' if c in numeric else str).reshape(-1, 1),
                                    dtype=tf.float32 if c in numeric else tf.string)
            for c in features.columns}

def as_dataset(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((feature_dict(frame), frame['target'].to_numpy('float32')))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.batch(64).prefetch(tf.data.AUTOTUNE)

inputs, encoded = {}, []
for column in numeric:
    inp = keras.Input(shape=(1,), name=column, dtype='float32')
    normalizer = layers.Normalization(axis=None)
    normalizer.adapt(train[column].to_numpy('float32'))
    inputs[column] = inp
    encoded.append(normalizer(inp))
for column in categorical:
    inp = keras.Input(shape=(1,), name=column, dtype='string')
    lookup = layers.StringLookup(output_mode='one_hot')
    lookup.adapt(tf.convert_to_tensor(train[column].to_numpy(str), dtype=tf.string))
    inputs[column] = inp
    encoded.append(lookup(inp))
print('Skaitiniai:', numeric)
print('Kategoriniai:', categorical)
train_ds, val_ds, test_ds = as_dataset(train, True), as_dataset(val), as_dataset(test)


### 3. Modelio sandara ir mokymas

Kaip originale, sujungti požymiai pereina per 64 ir 32 neuronų ReLU sluoksnius bei `Dropout`.
Vienas sigmoid išėjimas reiškia klasės „užsisakė indėlį“ tikimybės įvertį. Jam tinka įprastas
`binary_crossentropy`; nenaudojamas logits režimas. `EarlyStopping` pagal validavimo nuostolį
atkuria geriausius svorius. Retesnei teigiamai klasei mokant suteikiamas didesnis svoris.


In [ ]:
x = layers.Concatenate()(encoded)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dropout(0.3)(x)
model = keras.Model(inputs, layers.Dense(1, activation='sigmoid')(x))
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.AUC(name='auc')])
model.summary()
counts = train.target.value_counts()
weights = {int(k): len(train) / (2 * int(n)) for k, n in counts.items()}
history = model.fit(train_ds, validation_data=val_ds, epochs=30, class_weight=weights,
    callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
               keras.callbacks.TensorBoard(log_dir=str(OUT / 'tensorboard_nn'))], verbose=2)
pd.DataFrame(history.history).to_csv(OUT / 'nn_mokymo_eiga.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, metric in zip(axes, ['loss', 'auc']):
    ax.plot(history.history[metric], label='mokymas')
    ax.plot(history.history['val_' + metric], label='validavimas')
    ax.set(xlabel='Epocha (nuo 0)', title=metric)
    ax.legend()
plt.tight_layout(); plt.show()


### 4. Nepriklausomas testas, etalonas ir išsaugojimas

Kadangi klasių skaičiai skiriasi, vien accuracy neužtenka: rodomi precision, recall, F1,
ROC AUC ir average precision (PR kreivės santrauka). Paprastas etalonas visiems priskiria
mokymo daugumos klasę. F1 slenkstis parenkamas validavime; testas naudojamas tik galutiniam vertinimui.
`.keras` faile saugomas ir požymių apdorojimas. Tikriname, ar įkeltas modelis grąžina tuos pačius rezultatus.


In [ ]:
def binary_metrics(y, probabilities, threshold):
    predicted = (np.asarray(probabilities).ravel() >= threshold).astype(int)
    return {'accuracy': accuracy_score(y, predicted),
            'precision': precision_score(y, predicted, zero_division=0),
            'recall': recall_score(y, predicted, zero_division=0),
            'f1': f1_score(y, predicted, zero_division=0),
            'roc_auc': roc_auc_score(y, probabilities),
            'pr_auc_AP': average_precision_score(y, probabilities)}

def choose_threshold(y, probabilities):
    # Slenkstis pasirenkamas TIK validavimo rinkinyje. Testo rodikliai jo nekeičia.
    thresholds = np.linspace(0.05, 0.95, 91)
    scores = [f1_score(y, np.asarray(probabilities).ravel() >= t, zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(scores))])


In [ ]:
p_val = model.predict(feature_dict(val), verbose=0).ravel()
threshold = choose_threshold(val.target, p_val)
p_test = model.predict(feature_dict(test), verbose=0).ravel()
metrics = binary_metrics(test.target, p_test, threshold)
majority = int(train.target.mode().iloc[0])
baseline = binary_metrics(test.target, np.full(len(test), float(majority)), 0.5)
display(pd.DataFrame([baseline, metrics], index=['daugumos_etalonas', 'neuroninis_tinklas']))
ConfusionMatrixDisplay.from_predictions(test.target, p_test >= threshold,
    display_labels=['ne', 'taip'], cmap='Blues')
plt.title(f'Testas; slenkstis iš validavimo = {threshold:.2f}'); plt.show()
model.save(OUT / 'bank_nn.keras')
restored = keras.models.load_model(OUT / 'bank_nn.keras')
np.testing.assert_allclose(p_test, restored.predict(feature_dict(test), verbose=0).ravel(), atol=1e-5)
(OUT / 'nn_rodikliai.json').write_text(json.dumps({'threshold': threshold, **metrics}, indent=2))
sample = test.drop(columns='target').iloc[[0]]
display(sample)
print('Tikimybės įvertis:', float(restored.predict(feature_dict(sample), verbose=0)[0, 0]),
      '| Tikroji klasė:', int(test.target.iloc[0]))
print(f'Testo F1={metrics["f1"]:.3f}; AP={metrics["pr_auc_AP"]:.3f}; '
      f'etaloninis accuracy={baseline["accuracy"]:.3f}.')
print('Įkėlus modelį prognozės sutapo. Rodikliai apibūdina tik šį duomenų padalijimą.')


### 5. Paaiškinimas ir ribos

Požymiai → skaitinimas ir normalizavimas → neuroninis tinklas → indėlio tikimybės įvertis.
Mažas validavimo nuostolis pats savaime negarantuoja geros recall ar kalibruotų tikimybių.
Grafike didėjantis atotrūkis tarp mokymo ir validavimo gali rodyti persimokymą.
F1 slenkstis yra mokomoji taisyklė: realiame banke jį reikėtų sieti su kampanijos kaštais.
Kitame 1.3 skyriuje tas pats duomenų padalijimas leidžia nagrinėti alternatyvų algoritmą.

Papildoma dokumentacija: [TensorFlow struktūrinių duomenų klasifikavimas](https://www.tensorflow.org/tutorials/structured_data/preprocessing_layers).


## 1.3.2. TensorFlow Decision Forests banko klientams

Pritaikyta pagal dėstytojo pateiktą `AdventureWorksDW_Decision_Forests.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Išlaikytas tikras TensorFlow Decision Forests modelis, medžio vaizdavimas, OOB analizė, vertinimas ir išsaugojimas.


### 2. Suderinama TFDF aplinka

TFDF turi TensorFlow versijai jautrius C++ plėtinius. [Oficialus 1.12.0 leidimas](https://github.com/tensorflow/decision-forests/releases/tag/v1.12.0)
skirtas TensorFlow 2.19.0 ir palaiko Python 3.12. Todėl Colab Linux aplinkoje sukuriama izoliuota
Python 3.12 aplinka; pagrindinio Colab TensorFlow perrašyti ir sesijos perkrauti nereikia.
Pirmas įdiegimas atsisiunčia nemažai priklausomybių. Tai įprastas TFDF mokymas CPU, GPU jam nereikalingas.
Originale naudotas `dtreeviz` blokas buvo nepilnai sujungtas; čia medį rodo paties TFDF HTML.


In [ ]:
%pip -q install uv


In [ ]:
import sys, subprocess, platform
assert platform.system() == 'Linux', 'Šį TFDF užrašyną paleiskite Google Colab (Linux).'
ENV = Path('/content/nd1_tfdf_py312')
PY = ENV / 'bin/python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(ENV)], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(PY),
    'tensorflow==2.19.0', 'tf-keras==2.19.0', 'tensorflow-decision-forests==1.12.0',
    'ydf==0.11.0', 'protobuf>=5.29.1,<6',
    'pandas>=2.2,<3', 'scikit-learn>=1.5,<2', 'numpy>=1.26,<2.2'], check=True)


### 3. Miško mokymas, tikrinimas ir interpretavimas

Originalo dešimties medžių miškas pakeistas į 100 medžių, iki 12 lygių gylio.
Miškas pats apdoroja kategorinius požymius, todėl one-hot ir normalizavimas nereikalingi.
`fit` augina medžius vienu procesu; tai nėra epochomis mokomas neuroninis tinklas.

Žemiau esantis skriptas aiškiai atskiria mokymą, slenksčio pasirinkimą validavime ir testą.
Jis taip pat išsaugo OOB (mokymo įrašų, nepatekusių į konkrečių medžių imtis) statistiką,
požymių svarbą ir medį. OOB vertinimas nepakeičia galutinio nepriklausomo testo.


In [ ]:
TRAIN_SCRIPT = """
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras
import tensorflow_decision_forests as tfdf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
out = Path('rezultatai/1_3')
train, val, test = [pd.read_csv(out / f'bank_{s}.csv') for s in ['train', 'val', 'test']]
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train, label='target')
forest = tfdf.keras.RandomForestModel(num_trees=100, max_depth=12,
    min_examples=5, random_seed=42, verbose=1)
forest.fit(train_ds)
forest.compile(metrics=['accuracy'])
evaluation = forest.evaluate(tfdf.keras.pd_dataframe_to_tf_dataset(test, label='target'), return_dict=True)
print('TFDF testas:', evaluation)
# Prognozės duomenų rinkinyje nėra target stulpelio.
def ds(frame):
    return tfdf.keras.pd_dataframe_to_tf_dataset(frame.drop(columns='target'))
pv = forest.predict(ds(val), verbose=0).ravel()
pt = forest.predict(ds(test), verbose=0).ravel()
thresholds = np.linspace(0.05, 0.95, 91)
scores = [f1_score(val.target, pv >= t, zero_division=0) for t in thresholds]
threshold = float(thresholds[int(np.argmax(scores))])
pred = pt >= threshold
metrics = dict(threshold=threshold, accuracy=accuracy_score(test.target, pred),
    precision=precision_score(test.target, pred, zero_division=0),
    recall=recall_score(test.target, pred, zero_division=0), f1=f1_score(test.target, pred),
    roc_auc=roc_auc_score(test.target, pt), pr_auc_AP=average_precision_score(test.target, pt))
(out / 'df_rodikliai.json').write_text(json.dumps(metrics, indent=2))
pd.DataFrame({'target': test.target, 'probability': pt, 'prediction': pred.astype(int)}).to_csv(out / 'df_prognozes.csv', index=False)
inspector = forest.make_inspector()
logs = inspector.training_logs()
pd.DataFrame([{'trees': x.num_trees, 'oob_accuracy': x.evaluation.accuracy,
               'oob_loss': x.evaluation.loss} for x in logs]).to_csv(out / 'df_oob.csv', index=False)
# Vienas medis paaiškina vieną miško dalį, o ne visą kolektyvinę prognozę.
html = tfdf.model_plotter.plot_model(forest, tree_idx=0, max_depth=3)
(out / 'df_medis.html').write_text(html, encoding='utf-8')
importance = inspector.variable_importances()
key = 'NUM_AS_ROOT' if 'NUM_AS_ROOT' in importance else next(iter(importance))
pd.DataFrame([{'feature': f.name, 'importance': value} for f, value in importance[key]]).to_csv(out / 'df_svarba.csv', index=False)
(out / 'df_svarbos_tipas.txt').write_text(key)
# TFDF naudoja su TF 2.19 suderinamą tf_keras, ne Keras 3 .keras formatą.
forest.save(str(out / 'bank_forest'), overwrite=True)
restored = tf_keras.models.load_model(str(out / 'bank_forest'))
np.testing.assert_allclose(pt, restored.predict(ds(test), verbose=0).ravel(), atol=1e-6)
print('Išsaugoto ir įkelto miško prognozės sutapo.')
print(json.dumps(metrics, indent=2))
"""
SCRIPT = OUT / "mokyti_tfdf.py"
SCRIPT.write_text(TRAIN_SCRIPT, encoding="utf-8")
subprocess.run([str(PY), str(SCRIPT)], check=True)


### 4. Rezultatai ir grafikai

Tikslumas lyginamas su daugumos klasės etalonu. Klasių disbalansui vertinti pateikiami ir F1 bei AP.
Požymių svarba apibūdina modelio elgesį; ji neįrodo priežastinio poveikio indėlio pasirinkimui.
Interaktyviame medyje sąlygos viršuje skaido klientus į grupes. Lapai turi klasių pasiskirstymus.


In [ ]:
from IPython.display import HTML
metrics = json.loads((OUT / 'df_rodikliai.json').read_text())
display(pd.DataFrame([metrics], index=['TFDF']))
print('Daugumos klasės etaloninis accuracy:', (test.target == int(train.target.mode().iloc[0])).mean())
predictions = pd.read_csv(OUT / 'df_prognozes.csv')
ConfusionMatrixDisplay.from_predictions(predictions.target, predictions.prediction,
    display_labels=['ne', 'taip'], cmap='Greens')
plt.title('TFDF nepriklausomas testas'); plt.show()
logs = pd.read_csv(OUT / 'df_oob.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, metric in zip(axes, ['oob_accuracy', 'oob_loss']):
    ax.plot(logs.trees, logs[metric]); ax.set(xlabel='Medžių skaičius', title=metric)
plt.tight_layout(); plt.show()
importance = pd.read_csv(OUT / 'df_svarba.csv').sort_values('importance')
importance.plot.barh(x='feature', y='importance', legend=False, figsize=(7, 5))
plt.title('Požymių svarba: ' + (OUT / 'df_svarbos_tipas.txt').read_text()); plt.tight_layout(); plt.show()
display(HTML((OUT / 'df_medis.html').read_text(encoding='utf-8')))
nn_file = OUT / 'nn_rodikliai.json'
if nn_file.exists():
    display(pd.DataFrame([json.loads(nn_file.read_text()), metrics], index=['NN', 'TFDF']))
else:
    print('NN rodiklių šioje sesijoje nėra; jie pateikiami 1.3.1 užrašyne.')
print(f'TFDF testo F1={metrics["f1"]:.3f}, ROC AUC={metrics["roc_auc"]:.3f}.')


### 5. Ką parodo palyginimas?

NN mokosi skaitinių svorių, o miškas sujungia daugelio sprendimų medžių išvadas.
Abiejuose 1.3 skyriuose naudojamos tos pačios 8 000 eilučių parinkimo ir 70/15/15 skaidymo taisyklės.
Sąžiningai lyginti reikia abiejų skyrių tikrus testo rodiklius, o ne vien mokymo accuracy.
Nėra garantijos, kad sudėtingesnis algoritmas bus geresnis. OOB kreivės stabilizavimasis rodo,
ar papildomi medžiai dar keičia vidinį įvertį.

Šaltiniai: [TFDF pradžiamokslis](https://www.tensorflow.org/decision_forests/tutorials/beginner_colab),
[diegimas ir suderinamumas](https://www.tensorflow.org/decision_forests/installation).


In [ ]:
OUT = RESULTS_ROOT / '1_4'
OUT.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)


## 1.4. CNN gėlių nuotraukų klasifikavimui

Pritaikyta pagal dėstytojo pateiktą `cnn.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Išsaugota CNN ir MLP mokymo eiga, filtrų bei požymių žemėlapių analizė ir rankinis konvoliucijos skaičiavimas.


### 1. Pasirinkti vaizdų duomenys

Originalo CIFAR-10 / interneto nuotraukos pakeistos **Flowers Photos** rinkiniu:
`daisy`, `dandelion`, `roses`, `sunflowers`, `tulips` (5 klasės).
[TensorFlow šaltinis](https://www.tensorflow.org/tutorials/load_data/images).
Visas archyvas užima apie 218 MB. Nuotraukų autoriai ir CC BY licencijos pateikti archyvo `LICENSE.txt`.
Tai vieši pasirinkti duomenys, o ne mano paties fotografuotos nuotraukos.

Eksperimentui atsitiktinai pasirenkama iki 200 nuotraukų iš kiekvienos klasės.
Pasikartojantys vienodi failų turiniai atmetami prieš skaidymą. Mokymo, validavimo ir testo dalys:
70 %, 15 %, 15 %. Visuose trijuose vaizdų skyriuose parinkimo taisyklės vienodos.
Panašių, bet ne identiškų nuotraukų šis patikrinimas neaptinka.


In [ ]:
from PIL import Image, ImageDraw
from collections import Counter
import tarfile
CACHE = Path('/content/nd1_duomenys') if Path('/content').exists() else Path('nd1_duomenys')
CACHE.mkdir(exist_ok=True)
archive = keras.utils.get_file(fname='flower_photos.tgz',
    origin='https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
    cache_dir=str(CACHE.resolve()), cache_subdir='archyvai', extract=False)
DATA = CACHE / 'flower_photos'
if not DATA.exists():
    with tarfile.open(archive) as tar:
        tar.extractall(CACHE, filter='data')
classes = sorted(p.name for p in DATA.iterdir() if p.is_dir())
assert classes == ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
rng = np.random.default_rng(SEED)
rows, seen = [], set()
for label, name in enumerate(classes):
    files = sorted((DATA / name).glob('*.jpg'))
    # Atsitiktinimas atliekamas po rūšiavimo, kad failų sistemos tvarka nekeistų imties.
    files = [files[int(i)] for i in rng.permutation(len(files))]
    selected = 0
    for path in files:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        rows.append({'path': str(path), 'relative_path': str(path.relative_to(DATA)),
                     'label': label, 'class_name': name, 'sha256': digest})
        selected += 1
        if selected == 200:
            break
images = pd.DataFrame(rows)
train_img, remaining = train_test_split(images, test_size=.30, stratify=images.label, random_state=SEED)
val_img, test_img = train_test_split(remaining, test_size=.50, stratify=remaining.label, random_state=SEED)
assert set(train_img.sha256).isdisjoint(val_img.sha256)
assert set(train_img.sha256).isdisjoint(test_img.sha256)
assert set(val_img.sha256).isdisjoint(test_img.sha256)
manifest = pd.concat([part.assign(split=name) for name, part in
    [('train', train_img), ('val', val_img), ('test', test_img)]])
manifest.drop(columns='path').to_csv(OUT / 'geliu_padalinimas.csv', index=False)
display(pd.crosstab(manifest.class_name, manifest.split))
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, name in zip(axes, classes):
    path = train_img.loc[train_img.class_name == name, 'path'].iloc[0]
    ax.imshow(Image.open(path).convert('RGB')); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()


### 2. Įvestis ir konvoliucinis tinklas

Originale buvo 32×32 CIFAR-10 vaizdai ir 10 klasių, čia 96×96 RGB vaizdai ir 5 klasės.
`Rescaling` reikšmes 0..255 pakeičia į 0..1. `Conv2D` mokosi vietinių raštų, o `MaxPooling2D`
mažina žemėlapių dydį. `Flatten` išskleidžia požymius, tankieji sluoksniai juos paverčia klasėmis.
Paskutinis softmax sluoksnis grąžina penkis tikimybių įverčius; etiketės yra sveikieji 0..4.


In [ ]:
SIZE = 96  # Vienoda įvestis CNN ir MLP palyginimui.
BATCH = 32
def image_dataset(frame, training=False, size=SIZE):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(str), frame.label.to_numpy('int32')))
    def decode(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        # Ši funkcija palieka reikšmes 0..255; reikiamas mastelis nustatomas modelyje.
        return tf.image.resize(tf.cast(image, tf.float32), [size, size]), label
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = image_dataset(train_img, True), image_dataset(val_img), image_dataset(test_img)


In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(SIZE, SIZE, 3)), layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu', name='first_conv'), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'), layers.Flatten(),
    layers.Dense(64, activation='relu'), layers.Dropout(.3),
    layers.Dense(len(classes), activation='softmax')], name='flowers_cnn')
cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()
history_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=15,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)], verbose=2)
pd.DataFrame(history_cnn.history).to_csv(OUT / 'cnn_mokymo_eiga.csv', index=False)


In [ ]:
def image_metrics(name, model):
    # Testo duomenų tvarka nekeičiama, todėl sutampa su test_img etiketėmis.
    probability = model.predict(test_ds, verbose=0)
    predicted = probability.argmax(axis=1)
    y = test_img.label.to_numpy()
    result = {'modelis': name, 'accuracy': accuracy_score(y, predicted),
              'macro_f1': f1_score(y, predicted, average='macro', zero_division=0)}
    print(classification_report(y, predicted, labels=range(len(classes)), target_names=classes, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y, predicted, labels=range(len(classes)),
        display_labels=classes, xticks_rotation=35, cmap='Blues')
    plt.title(name + ': nepriklausomas testas'); plt.tight_layout(); plt.show()
    return result

def show_history(history, title):
    frame = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, metric in zip(axes, ['loss', 'accuracy']):
        ax.plot(frame[metric], label='mokymas')
        ax.plot(frame['val_' + metric], label='validavimas')
        ax.set(xlabel='Epocha (nuo 0)', title=title + ': ' + metric); ax.legend()
    plt.tight_layout(); plt.show()


In [ ]:
show_history(history_cnn, 'CNN')


### 3. Filtrai ir požymių žemėlapiai

Filtras yra išmoktas 3×3×3 svorių rinkinys. Požymių žemėlapis gaunamas slenkant filtrą per vaizdą,
pridedant poslinkį ir taikant ReLU. Žemiau rodomi 16 iš 32 filtrų ir jų aktyvacijos vienai testo nuotraukai.
Filtrai normalizuojami tik vaizdavimui; modelio svoriai nekeičiami.


In [ ]:
sample_images, sample_labels = next(iter(test_ds))
sample = sample_images[:1]
conv = cnn.get_layer('first_conv')
kernels, biases = conv.get_weights()
probe = keras.Model(cnn.inputs, conv.output)
maps = probe(sample, training=False).numpy()
fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for i in range(16):
    kernel = kernels[:, :, :, i]
    axes.flat[i].imshow((kernel - kernel.min()) / (np.ptp(kernel) + 1e-8))
    axes.flat[i].set_title(f'Filtras {i}'); axes.flat[i].axis('off')
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for i in range(16):
    axes.flat[i].imshow(maps[0, :, :, i], cmap='viridis')
    axes.flat[i].set_title(f'Žemėlapis {i}'); axes.flat[i].axis('off')
plt.tight_layout(); plt.show()


### 4. Rankinis pirmo sluoksnio skaičiavimas

Kaip originale, vienam filtrui apskaičiuojama 6×6 išvesties sritis.
Keras `Conv2D` iš tiesų atlieka kryžminę koreliaciją (branduolys neapverčiamas).
Vieno taško reikšmė: ReLU(įvesties 3×3×3 lopinėlio ir filtro sandaugų suma + poslinkis).
Svarbu naudoti įvestį po `Rescaling`. `assert_allclose` patikrina sutapimą su tikru sluoksniu.


In [ ]:
scaled = sample.numpy()[0] / 255.0
filter_index = 2
manual = np.zeros((6, 6), dtype='float32')
for row in range(6):
    for column in range(6):
        patch = scaled[row:row+3, column:column+3, :]
        manual[row, column] = max(0.0, float(np.sum(patch * kernels[:, :, :, filter_index]) + biases[filter_index]))
actual = maps[0, :6, :6, filter_index]
np.testing.assert_allclose(manual, actual, atol=1e-4, rtol=1e-4)
print('Rankinė koreliacija sutapo; didžiausias skirtumas:', float(np.max(np.abs(manual - actual))))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(manual); axes[0].set_title('Rankinis skaičiavimas')
axes[1].imshow(actual); axes[1].set_title('Keras Conv2D')
plt.tight_layout(); plt.show()


### 5. MLP palyginimas

Originalo MLP irgi mokomas tais pačiais vaizdais. Jis iškart išskleidžia visus pikselius,
todėl neturi CNN vietinių filtrų ir svorių bendrinimo. Palyginimui naudojami tie patys
vaizdų dydžiai, padalijimas, maksimali epochų riba ir ankstyvo stabdymo taisyklė.
Parametrų skaičiai skiriasi, todėl tai architektūrų mokomasis, o ne griežtai vienodo biudžeto tyrimas.


In [ ]:
mlp = keras.Sequential([keras.Input(shape=(SIZE, SIZE, 3)), layers.Rescaling(1./255), layers.Flatten(),
    layers.Dense(32, activation='relu'), layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'), layers.Dense(len(classes), activation='softmax')], name='flowers_mlp')
mlp.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.summary()
history_mlp = mlp.fit(train_ds, validation_data=val_ds, epochs=15,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)], verbose=2)
show_history(history_mlp, 'MLP')
results = pd.DataFrame([image_metrics('CNN', cnn), image_metrics('MLP', mlp)])
results['parametrai'] = [cnn.count_params(), mlp.count_params()]
display(results)
results.to_csv(OUT / 'cnn_mlp_testas.csv', index=False)
cnn.save(OUT / 'flowers_cnn.keras')
reloaded = keras.models.load_model(OUT / 'flowers_cnn.keras')
np.testing.assert_allclose(cnn(sample, training=False).numpy(), reloaded(sample, training=False).numpy(), atol=1e-5)
display(pd.DataFrame(cnn(sample, training=False).numpy(), columns=classes))
winner = results.loc[results.macro_f1.idxmax()]
print(f'Šiame padalijime didžiausias macro F1: {winner["modelis"]}, {winner["macro_f1"]:.3f}.')


### 6. Rezultatų interpretacija

Macro F1 vienodai įvertina kiekvienos klasės F1, o painiavos matrica parodo, kurios gėlės painiojamos.
CNN nėra iš anksto paskelbiamas geresniu: išvada apskaičiuojama pagal tikrą šio paleidimo testą.
Iš vienos imties negalima spręsti apie visas nuotraukų sąlygas. „Teisinga“ softmax suma
nereiškia, kad prognozė teisinga arba tikimybė kalibruota.

Šaltiniai: [TensorFlow CNN](https://www.tensorflow.org/tutorials/images/cnn),
[Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D).


In [ ]:
import gc
del cnn, mlp, reloaded, probe
keras.backend.clear_session()
gc.collect()
OUT = RESULTS_ROOT / '1_5_1'
OUT.mkdir(parents=True, exist_ok=True)
manifest.drop(columns='path').to_csv(OUT / 'geliu_padalinimas.csv', index=False)


Gėlių duomenys ir jų padalijimas jau paruošti 1.4 dalyje. Jie pakartotinai neatsisiunčiami ir neskaidomi.


## 1.5.1. Iš anksto apmokytų modelių pritaikymas nuotraukai

Pritaikyta pagal dėstytojo pateiktą `cnn_One_Image.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Išsaugoti klasifikavimas, iškirptos srities analizė, objektų aptikimas ir semantinis segmentavimas. Naudojama parinkta gėlių rinkinio nuotrauka.


### 2. Du klasifikatoriai ir jų įvestys

Originalas lygino 21 tūkst. ir 1 tūkst. klasių TF Hub modelius; čia naudojami Keras
`MobileNetV2` ir `EfficientNetB0`, abu apmokyti ImageNet 1000 klasių rinkiniu.
Tai leidžia abiem taikyti tą patį patikimą `decode_predictions` žodyną.
MobileNet įvestį reikia perkelti į −1..1, o EfficientNet turi savo mastelio keitimo sluoksnį
ir gauna 0..255 reikšmes. Šiame skyriuje svoriai **nepermokomi**: tai parengto modelio panaudojimas.
Naujos 5 gėlių klasės iš tiesų išmokomos kitame, 1.5.2 skyriuje.


In [ ]:
photo_path = test_img.loc[test_img.class_name == 'sunflowers', 'path'].iloc[0]
photo = Image.open(photo_path).convert('RGB')
plt.figure(figsize=(6, 5)); plt.imshow(photo); plt.title('Pasirinkta testo nuotrauka: sunflowers'); plt.axis('off'); plt.show()
mobile = keras.applications.MobileNetV2(weights='imagenet')
efficient = keras.applications.EfficientNetB0(weights='imagenet')

def classify(pil_image):
    array = np.asarray(pil_image.resize((224, 224)), dtype='float32')[None]
    results = []
    for name, net, preprocessing in [
        ('MobileNetV2', mobile, keras.applications.mobilenet_v2.preprocess_input),
        ('EfficientNetB0', efficient, keras.applications.efficientnet.preprocess_input)]:
        probabilities = net.predict(preprocessing(array.copy()), verbose=0)
        # Abu modeliai jau turi softmax. Antrą kartą softmax netaikomas.
        decoded = keras.applications.imagenet_utils.decode_predictions(probabilities, top=5)[0]
        results.extend({'modelis': name, 'klasė_ImageNet': label, 'tikimybės_įvertis': float(p)}
                       for _, label, p in decoded)
    return pd.DataFrame(results)

whole_predictions = classify(photo)
display(whole_predictions)
whole_predictions.to_csv(OUT / 'isankstiniu_modeliu_top5.csv', index=False)


### 3. Iškirptos srities įtaka

Originale kadravimas keitė, ką modelis mato. Čia iškerpama centrinė nuotraukos sritis:
paliekama 60 % pločio ir aukščio. Tai nekeičia etikečių ir modelio, bet keičia įvesties kontekstą.
Lyginamas top-5 sąrašas; neskelbiama accuracy, nes rinkinio `sunflowers` klasė nėra bendras
1000 klasių ImageNet testas.


In [ ]:
w, h = photo.size
crop = photo.crop((int(.2*w), int(.2*h), int(.8*w), int(.8*h)))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(photo); axes[0].set_title('Visa nuotrauka'); axes[0].axis('off')
axes[1].imshow(crop); axes[1].set_title('Centrinė iškarpa'); axes[1].axis('off')
plt.tight_layout(); plt.show()
display(classify(crop))
# Atlaisvinama klasifikatorių atmintis prieš didesnius vaizdų analizės modelius.
del mobile, efficient
keras.backend.clear_session()
import gc
gc.collect()


### 4. Objektų aptikimas

Kaip originale, naudojamas iš anksto apmokytas Open Images SSD MobileNetV2 detektorius.
Jis grąžina ne visos nuotraukos klasę, o objektų stačiakampius ir jų pasitikėjimo įverčius.
Koordinatės pateikiamos normalizuota tvarka `(ymin, xmin, ymax, xmax)` ir padauginamos iš nuotraukos dydžio.
Rodomi tik įverčiai ≥ 0,25; jei tokių nėra, tai aiškiai parašoma.

SavedModel atsisiunčiamas pagal [oficialų TF Hub archyvų protokolą](https://www.tensorflow.org/hub/hosting)
ir įkeliamas su `tf.saved_model.load`. Taip nereikia maišyti senesnio `hub.KerasLayer` ir Keras 3.


In [ ]:
def load_saved_hub(handle, name):
    folder = CACHE / name
    if not (folder / 'saved_model.pb').exists():
        archive_path = keras.utils.get_file(fname=name + '.tar.gz',
            origin=handle + '?tf-hub-format=compressed',
            cache_dir=str(CACHE.resolve()), cache_subdir='modeliai', extract=False)
        folder.mkdir(exist_ok=True)
        if not tarfile.is_tarfile(archive_path):
            raise ValueError('Modelio šaltinis negrąžino tar archyvo. Patikrinkite modelio prieinamumą.')
        with tarfile.open(archive_path) as tar:
            tar.extractall(folder, filter='data')
    return tf.saved_model.load(str(folder))

detector = load_saved_hub('https://tfhub.dev/google/openimages_v4/ssd/mobilenet_v2/1', 'ssd_openimages')
detect = detector.signatures['default']
# Aptikimas vykdomas sumažinta nuotrauka; santykinės koordinatės tinka ir originalui.
small = photo.copy()
small.thumbnail((640, 640))
detected = detect(tf.convert_to_tensor(np.asarray(small, dtype='float32')[None] / 255.0))
boxes = detected['detection_boxes'].numpy()
scores = detected['detection_scores'].numpy()
labels = detected['detection_class_entities'].numpy()
annotated = photo.copy()
draw = ImageDraw.Draw(annotated)
found = []
for box, score, label in zip(boxes, scores, labels):
    if score < .25:
        continue
    y0, x0, y1, x1 = box
    label = label.decode('utf-8') if isinstance(label, bytes) else str(label)
    draw.rectangle((x0*w, y0*h, x1*w, y1*h), outline='red', width=3)
    draw.text((x0*w, y0*h), f'{label} {score:.2f}', fill='red')
    found.append({'objektas': label, 'įvertis': float(score)})
display(pd.DataFrame(found))
if not found:
    print('Nėra objektų, viršijančių 0,25 slenkstį; gėlės aptikimas nėra garantuotas.')
plt.figure(figsize=(8, 6)); plt.imshow(annotated); plt.axis('off'); plt.title('Objektų aptikimas'); plt.show()
annotated.save(OUT / 'objektu_aptikimas.png')
del detector, detect, detected
gc.collect()


### 5. Semantinis segmentavimas

HRNet, apmokytas ADE20K, priskiria klasę kiekvienam pikseliui. Tai skiriasi nuo objektų
stačiakampių aptikimo. Šis modelis turi 151 kanalą: foną / ignoruojamą klasę ir 150 ADE20K klasių.
Kad nebūtų klaidingai sujungti skirtingai indeksuoti klasių žodynai, pateikiami modelio klasių ID.
Modelio įvestis yra RGB 0..1; sumažinama iki 256×256, kad reikėtų mažiau atminties.
Pikselių dalis nereiškia segmentavimo tikslumo – tam reikėtų teisingų pikselių kaukių.


In [ ]:
segmenter = load_saved_hub('https://tfhub.dev/google/HRNet/ade20k-hrnetv2-w48/1', 'hrnet_ade20k')
seg_input = np.asarray(photo.resize((256, 256)), dtype='float32') / 255.0
# Šis SavedModel eksportuoja predict metodą, kaip oficialiame HRNet pavyzdyje.
seg_scores = segmenter.predict(tf.convert_to_tensor(seg_input[None]))
seg_scores = np.asarray(seg_scores)
assert seg_scores.ndim == 4 and seg_scores.shape[0] == 1
mask = np.argmax(seg_scores[0], axis=-1)
ids, counts = np.unique(mask, return_counts=True)
distribution = pd.DataFrame({'klases_ID': ids, 'pikseliu_dalis': counts / mask.size}).sort_values('pikseliu_dalis', ascending=False)
display(distribution.head(10))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(seg_input); axes[0].set_title('Įvestis')
axes[1].imshow(mask, cmap='nipy_spectral', vmin=0, vmax=150); axes[1].set_title('Kiekvieno pikselio klasė')
top_id = int(distribution.klases_ID.iloc[0])
axes[2].imshow(mask == top_id, cmap='gray'); axes[2].set_title(f'Didžiausia sritis: ID {top_id}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
np.save(OUT / 'segmentavimo_klasiu_kauke.npy', mask)
distribution.to_csv(OUT / 'segmentavimo_pikseliu_dalys.csv', index=False)
print('Klasifikavimas aprašo visą vaizdą; aptikimas randa stačiakampius; segmentavimas žymi pikselius.')


### 6. Pritaikymo išvada

Iš anksto apmokyti modeliai panaudoti naujai pasirinktai nuotraukai ir jos iškarpai.
Klasių sąrašai skirtingi: ImageNet, Open Images ir ADE20K. Todėl jų ID ir pasitikėjimų
negalima tiesiogiai lyginti tarpusavyje. Gėlė gali būti neatpažinta arba priskirta kitai klasei.
Šis pavyzdys rodo jau išmoktų reprezentacijų pritaikymą; kitame skyriuje atliekamas tikras papildomas mokymas.

Šaltiniai: [objektų aptikimas](https://www.tensorflow.org/hub/tutorials/object_detection),
[HRNet ir jo predict metodas](https://www.tensorflow.org/hub/tutorials/hrnet_semantic_segmentation),
[Keras Applications](https://keras.io/api/applications/).


In [ ]:
del segmenter, seg_scores
keras.backend.clear_session()
gc.collect()
OUT = RESULTS_ROOT / '1_5_2'
OUT.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
manifest.drop(columns='path').to_csv(OUT / 'geliu_padalinimas.csv', index=False)


Toliau naudojamos 1.4 dalyje sukurtos `image_dataset`, `image_metrics` ir `show_history` funkcijos. Įvesties dydis perkėlimo mokymuisi nustatomas atskirai.


## 1.5.2. Perkėlimo mokymasis ir papildomas mokymas gėlėms

Pritaikyta pagal dėstytojo pateiktą `cnn_Transfer_learning.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. CIFAR-10 pakeistas gėlių rinkiniu. Išlaikyti užšaldytas požymių išskyrimas, naujos galvos mokymas, išsaugojimas, įkėlimas, duomenų augmentavimas ir fine-tuning.


### 2. Įvestis ir duomenų augmentavimas

Originalo TF Hub EfficientNet sluoksnis pakeistas Keras MobileNetV2: modelis lengvesnis ir
nesukelia Keras 2 / Keras 3 maišymo. Naudojami ImageNet svoriai, tačiau 1000 klasių galva pašalinama.
Vaizdai yra 160×160 RGB. Apvertimas, pasukimas ir priartinimas atliekami tik mokymo metu;
validavimo ir testo vaizdai neaugmentuojami. Nauja galva turi 5 gėlių klases.


In [ ]:
SIZE = 160
train_ds = image_dataset(train_img, True, size=SIZE)
val_ds = image_dataset(val_img, size=SIZE)
test_ds = image_dataset(test_img, size=SIZE)
augmentation = keras.Sequential([layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomRotation(.08, seed=SEED+1), layers.RandomZoom(.10, seed=SEED+2)], name='augmentavimas')
batch, _ = next(iter(train_ds))
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax in axes:
    ax.imshow(np.clip(augmentation(batch[:1], training=True).numpy()[0] / 255.0, 0, 1)); ax.axis('off')
plt.suptitle('Vienos mokymo nuotraukos variantai'); plt.tight_layout(); plt.show()
base = keras.applications.MobileNetV2(input_shape=(SIZE, SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False
inputs = keras.Input(shape=(SIZE, SIZE, 3))
x = augmentation(inputs)
x = layers.Rescaling(1./127.5, offset=-1)(x)  # MobileNetV2 įvestis: -1..1.
x = base(x, training=False)  # BatchNormalization statistika išsaugoma ir fine-tuning metu.
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(.2)(x)
outputs = layers.Dense(len(classes), activation='softmax')(x)
model = keras.Model(inputs, outputs, name='flowers_transfer')
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
print('Mokomų parametrų:', sum(int(np.prod(v.shape)) for v in model.trainable_weights))


### 3. Naujos galvos mokymas ir išsaugojimas

Užšaldytas bazinis tinklas išskiria bendrus vaizdo požymius; mokomi tik naujos galvos svoriai.
Validavimas naudojamas ankstyvam stabdymui ir geriausios epochos parinkimui.
Originale modelis buvo saugomas ir įkeliamas; čia naudojamas vientisas `.keras` formatas.
Iškart patikrinama, ar įkėlus prognozės sutampa.


In [ ]:
history_head = model.fit(train_ds, validation_data=val_ds, epochs=6,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=2)
model.save(OUT / 'flowers_frozen.keras')
frozen = keras.models.load_model(OUT / 'flowers_frozen.keras')
sample, _ = next(iter(val_ds))
np.testing.assert_allclose(model(sample, training=False).numpy(), frozen(sample, training=False).numpy(), atol=1e-5)
frozen_val_loss = float(frozen.evaluate(val_ds, verbose=0)[0])
print('Užšaldytas modelis įkeltas; prognozės sutapo. Validavimo loss:', frozen_val_loss)


In [ ]:
show_history(history_head, 'Užšaldyta bazė')


### 4. Fine-tuning: dalies bazės papildomas mokymas

Atšaldomi paskutiniai 20 bazinio tinklo sluoksnių, išskyrus `BatchNormalization`.
Pakeitus `trainable`, būtina **iš naujo compile**. Mokymosi žingsnis sumažinamas nuo 0,001 iki 0,00001,
kad neišardytume išmoktų požymių. Originalo tiesioginio `optimizer.lr` keitimo čia nereikia.
Tai atskira antroji mokymo fazė; ankstyvas stabdymas ir vėl naudoja tik validavimą.


In [ ]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Mokomų parametrų po atšaldymo:', sum(int(np.prod(v.shape)) for v in model.trainable_weights))
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=2)
show_history(history_fine, 'Fine-tuning')
fine_val_loss = float(model.evaluate(val_ds, verbose=0)[0])
selected = model if fine_val_loss < frozen_val_loss else frozen
selected_phase = 'fine_tuning' if fine_val_loss < frozen_val_loss else 'uzsaldyta_baze'
print('Modelis parinktas pagal validavimo loss:', selected_phase)
selected.save(OUT / 'flowers_galutinis.keras')
pd.DataFrame(history_head.history).to_csv(OUT / 'transfer_galvos_eiga.csv', index=False)
pd.DataFrame(history_fine.history).to_csv(OUT / 'transfer_fine_eiga.csv', index=False)


### 5. Galutinis nepriklausomas testas

Abi fazės vertinamos tame pačiame teste, tačiau galutinis modelis jau pasirinktas pagal validavimą.
Jei fine-tuning nepagerina validavimo nuostolio, lieka užšaldytas variantas – pagerėjimas nėra garantuotas.
Testo rezultatai nekeičiami į modelio parinkimo taisyklę. Rodoma macro F1, accuracy, painiavos matricos ir prognozių pavyzdžiai.


In [ ]:
results = pd.DataFrame([image_metrics('Užšaldyta bazė', frozen), image_metrics('Fine-tuning', model)])
display(results)
results.to_csv(OUT / 'transfer_testas.csv', index=False)
final = keras.models.load_model(OUT / 'flowers_galutinis.keras')
np.testing.assert_allclose(selected(sample, training=False).numpy(), final(sample, training=False).numpy(), atol=1e-5)
test_batch, test_labels = next(iter(test_ds))
probabilities = final(test_batch[:8], training=False).numpy()
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    prediction = int(probabilities[i].argmax())
    truth = int(test_labels[i])
    ax.imshow(np.clip(test_batch[i].numpy() / 255.0, 0, 1))
    ax.set_title(f'Tikra: {classes[truth]}\nSpėta: {classes[prediction]} ({probabilities[i].max():.2f})', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()
(OUT / 'geliu_klases.json').write_text(json.dumps(classes, ensure_ascii=False), encoding='utf-8')
(OUT / 'pasirinkta_faze.json').write_text(json.dumps({'phase': selected_phase,
    'frozen_val_loss': frozen_val_loss, 'fine_val_loss': fine_val_loss}, indent=2))
print('Galutinis modelis:', selected_phase, '| Išsaugojimas ir įkėlimas patikrinti.')


### 6. Kuo skiriasi abu 1.5 skyriai?

1.5.1 panaudojo gatavus modelius, kurių svoriai nebuvo keičiami. Šiame 1.5.2 skyriuje
ImageNet požymiai perkeliami į naują penkių gėlių klasių užduotį: pirma mokoma galva,
paskui maža bazės dalis. Augmentavimas didina mokymo pavyzdžių įvairovę, bet nesukuria
naujų nepriklausomų testo duomenų. Rezultatai galioja šio rinkinio imčiai; kitokios aplinkos
nuotraukos ar kelios gėlės viename kadre gali duoti prastesnes prognozes.

Šaltiniai: [TensorFlow transfer learning ir fine-tuning](https://www.tensorflow.org/tutorials/images/transfer_learning),
[Keras MobileNetV2](https://keras.io/api/applications/mobilenet/).


## Darbo išsaugojimas

Paleidus visas ląsteles turi būti matomi abiejų banko modelių rodikliai, CNN ir MLP grafikai,
filtrai bei rankinio skaičiavimo patikra, nuotraukos analizė ir perkėlimo mokymosi rezultatai.

1. Patikrinti, ar nė viena ląstelė nesibaigė klaida.
2. Pasirinkti **File → Save a copy in GitHub** ir išsaugoti šio užrašyno kelyje su išvestimis.
3. Jei reikia išsaugoti apmokytus modelius ir CSV, iki Colab sesijos pabaigos atsisiųsti `rezultatai/` katalogą.
